In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.4 MB/s eta 0:00:0000:0100:01


In [3]:
from pathlib import Path
import shutil
import yaml

SOURCE_DIR = Path("/kaggle/input/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels")
OUTPUT_DIR = Path("/kaggle/working/cardd-3class")

TARGET_CLASSES = {
    0: 0,  # dent -> dent
    1: 1,  # scratch -> scratch
}

CLEAN_CLASS = 2

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

for split in ["train", "val", "test"]:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

image_files = [
    p for p in SOURCE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Found {len(image_files)} images.")

def get_split(image_path):
    parts = [part.lower() for part in image_path.parts]

    if "train" in parts:
        return "train"

    if "val" in parts or "valid" in parts or "validation" in parts:
        return "val"

    if "test" in parts:
        return "test"

    return "train"

def find_label_file(image_path, split):
    possible_paths = [
        image_path.with_suffix(".txt"),
        image_path.parent / "labels" / f"{image_path.stem}.txt",
        image_path.parent.parent / "labels" / f"{image_path.stem}.txt",
        SOURCE_DIR / "labels" / split / f"{image_path.stem}.txt",
        SOURCE_DIR / "labels" / f"{image_path.stem}.txt",
    ]

    for path in possible_paths:
        if path.exists():
            return path

    matches = list(SOURCE_DIR.rglob(f"{image_path.stem}.txt"))

    if matches:
        return matches[0]

    return None

stats = {
    "train": {
        "images": 0,
        "dent": 0,
        "scratch": 0,
        "clean": 0
    },
    "val": {
        "images": 0,
        "dent": 0,
        "scratch": 0,
        "clean": 0
    },
    "test": {
        "images": 0,
        "dent": 0,
        "scratch": 0,
        "clean": 0
    }
}

for image_path in image_files:
    split = get_split(image_path)

    relative_path = image_path.relative_to(SOURCE_DIR)
    unique_name = "_".join(relative_path.with_suffix("").parts)

    output_image = (
        OUTPUT_DIR
        / "images"
        / split
        / f"{unique_name}{image_path.suffix.lower()}"
    )

    output_label = (
        OUTPUT_DIR
        / "labels"
        / split
        / f"{unique_name}.txt"
    )

    shutil.copy2(image_path, output_image)

    label_path = find_label_file(image_path, split)

    filtered_labels = []

    if label_path:
        with open(label_path, "r") as f:
            for line in f:
                values = line.strip().split()

                if len(values) != 5:
                    continue

                original_class = int(values[0])

                if original_class in TARGET_CLASSES:
                    new_class = TARGET_CLASSES[original_class]
                    values[0] = str(new_class)
                    filtered_labels.append(" ".join(values))

    if filtered_labels:
        with open(output_label, "w") as f:
            f.write("\n".join(filtered_labels) + "\n")

        classes_in_image = {
            int(label.split()[0])
            for label in filtered_labels
        }

        if 0 in classes_in_image:
            stats[split]["dent"] += 1

        if 1 in classes_in_image:
            stats[split]["scratch"] += 1

    else:
        with open(output_label, "w") as f:
            f.write("2 0.5 0.5 1.0 1.0\n")

        stats[split]["clean"] += 1

    stats[split]["images"] += 1

data_yaml = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "dent",
        1: "scratch",
        2: "clean"
    },
    "nc": 3
}

with open(OUTPUT_DIR / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("\nDataset processing complete!")
print(f"Output directory: {OUTPUT_DIR}")

for split, data in stats.items():
    print(f"\n{split.upper()}")
    print(f"Images: {data['images']}")
    print(f"Images with dent: {data['dent']}")
    print(f"Images with scratch: {data['scratch']}")
    print(f"Images labelled clean: {data['clean']}")

print("\nData YAML:")
print((OUTPUT_DIR / "data.yaml").read_text())

Found 4000 images.

Dataset processing complete!
Output directory: /kaggle/working/cardd-3class

TRAIN
Images: 2816
Images with dent: 1242
Images with scratch: 1507
Images labelled clean: 715

VAL
Images: 810
Images with dent: 352
Images with scratch: 431
Images labelled clean: 207

TEST
Images: 374
Images with dent: 157
Images with scratch: 183
Images labelled clean: 104

Data YAML:
path: /kaggle/working/cardd-3class
train: images/train
val: images/val
test: images/test
names:
  0: dent
  1: scratch
  2: clean
nc: 3



In [4]:
from ultralytics import YOLO

model = YOLO("yolo26m.pt")

results = model.train(
    data="/kaggle/working/cardd-3class/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    project="/kaggle/working/runs",
    name="cardd_3class"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cardd-3class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flip

In [5]:
model = YOLO("/kaggle/working/runs/cardd_3class/weights/best.pt")

results = model.train(
    data="/kaggle/working/cardd-3class/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    project="/kaggle/working/runs",
    name="cardd_3class_second"
)

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cardd-3class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/cardd_3class/weights/best.pt, momentum=0.937, mosaic=1.0, multi_